Behavioral features are stronger confounders (larger SMDs) so they take priority in W. Demographics go in X where their role is to explain heterogeneity. 

In [0]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import cross_val_score

gold_df = spark.table("causal_project.gold_household").toPandas()
print(f"Shape: {gold_df.shape}")

In [0]:
gold_df['log_outcome'] = np.log1p(gold_df['avg_daily_campaign_spend'])

gold_encoded = gold_df.copy()

gold_encoded['age_group'] = gold_encoded['age_group'].map({
    'Age Group1': 1, 'Age Group2': 2, 'Age Group3': 3,
    'Age Group4': 4, 'Age Group5': 5, 'Age Group6': 6
})
gold_encoded['income_level'] = gold_encoded['income_level'].str.replace(
    'Level', '', regex=False
).astype(int)

gold_encoded['household_size'] = gold_encoded['household_size'].map({
    '1': 1, '2': 2, '3': 3, '4': 4, '5+': 5
})
gold_encoded['kid_count'] = gold_encoded['kid_count'].map({
    'None/Unknown': 0, '1': 1, '2': 2, '3+': 3
})
gold_encoded['home_ownership'] = gold_encoded['home_ownership'].map({
    'Renter': 0, 'Probable Renter': 1, 'Unknown': 2,
    'Probable Owner': 3, 'Homeowner': 4
})
gold_encoded['marital_status'] = gold_encoded['marital_status'].map({
    'X': 0, 'Y': 1, 'Z': 2
})


In [0]:
x_cols = [
    'age_group', 'marital_status', 'income_level',
    'household_size', 'home_ownership', 'kid_count'
]
w_cols = [
    'pre_avg_weekly_spend',
    'pre_avg_weekly_trips',
    'pre_coupon_usage_rate',
    'pre_loyalty_card_rate',
    'pre_department_breadth',
    'pre_active_weeks',
    'clean_window_days',
]

Y = gold_encoded['log_outcome'].values
T = gold_encoded['treatment'].values
X = gold_encoded[x_cols].values.astype(float)
W = gold_encoded[w_cols].values.astype(float)

print(f"Y: {Y.shape}, T: {T.shape}, X: {X.shape}, W: {W.shape}")
print(f"TypeA: {T.sum()}, TypeBC: {(1-T).sum()}")

In [0]:
mlflow.set_experiment("/Users/kayvan2024@gmail.com/causal_project")
with mlflow.start_run(run_name="CausalForestDML_v1"):
    model = CausalForestDML(
        model_y=GradientBoostingRegressor(n_estimators=200,
        max_depth=4,
        random_state=42),
        model_t=GradientBoostingClassifier(n_estimators=200,
        max_depth=4,
        random_state=42),
        discrete_treatment=True,
        cv=3,
        n_estimators=500,
        min_samples_leaf=20,   # conservative given n=760
        max_features="auto",
        random_state=42
        )
    model.fit(Y,T,X=X,W=W)
    ate_inf = model.ate_inference(X=X)
    summary = ate_inf.summary()
    print(summary)
    
    mlflow.log_params({
        "estimator":       "CausalForestDML",
        "n_estimators":    500,
        "min_samples_leaf": 20,
        "outcome":         "log1p(avg_daily_campaign_spend)",
        "treatment":       "TypeA=1 vs TypeBC=0",
        "n_obs":           len(Y),
        "n_treated":       int(T.sum()),
        "n_control":       int((1-T).sum()),
    })

    ci_lower, ci_upper = ate_inf.conf_int_mean()
    mlflow.log_metric("ate", ate_inf.mean_point)
    mlflow.log_metric("ate_ci_lower", ci_lower)
    mlflow.log_metric("ate_ci_upper", ci_upper)
    mlflow.log_metric("ate_pvalue", ate_inf.pvalue())

In [0]:
cate = model.effect(X)
print(f"Households with positive CATE: {(cate > 0).sum()} / {len(cate)}")
print(f"Mean CATE for positive group: {cate[cate > 0].mean():.3f}")
print(f"Mean CATE for negative group: {cate[cate < 0].mean():.3f}")

In [0]:
cate = model.effect(X)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(cate, bins=40, color="#5b8db8", edgecolor="white", linewidth=0.4)
ax.axvline(cate.mean(), color="#e07b54", linewidth=1.5,
           label=f"Mean CATE = {cate.mean():.3f}")
ax.axvline(0, color="#888888", linewidth=1, linestyle="--", label="Zero effect")
ax.set_xlabel("Estimated individual treatment effect (log scale)")
ax.set_ylabel("Number of households")
ax.set_title("CATE distribution — CausalForestDML")
ax.legend()
plt.tight_layout()
mlflow.log_figure(fig, "cate_distribution.png")
plt.show()

print(f"Households with positive CATE: "
      f"{(cate > 0).sum()} / {len(cate)} "
      f"({(cate > 0).mean()*100:.1f}%)")

In [0]:
# Which households benefit most from personalized campaigns?
cate_df = gold_df[['household_key','treatment',
                    'income_level','age_group',
                    'home_ownership','kid_count']].copy()
cate_df['cate'] = cate

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['income_level', 'age_group', 'kid_count']):
    group_means = cate_df.groupby(col)['cate'].mean().sort_index()
    group_means.plot(kind='bar', ax=ax, color="#f7c59f", edgecolor="#e07b54")
    ax.axhline(0, color="#888888", linewidth=0.8, linestyle="--")
    ax.set_title(f"Mean CATE by {col}")
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=30)

plt.suptitle("Treatment effect heterogeneity across demographic groups",
             fontsize=12)
plt.tight_layout()
mlflow.log_figure(fig, "cate_heterogeneity.png")
plt.show()

In [0]:
# Which X features drive CATE variation most?
importances = model.feature_importances_
importance_df = pd.DataFrame({
    'feature':    x_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importance_df['feature'], importance_df['importance'],
        color="#a8d5a2", edgecolor="#5b8db8")
ax.set_xlabel("Heterogeneity importance")
ax.set_title("Which features drive treatment effect heterogeneity?")
ax.invert_yaxis()
plt.tight_layout()
mlflow.log_figure(fig, "heterogeneity_importance.png")
plt.show()

mlflow.end_run()
print("MLflow run complete.")

### CATE Results

All individual treatment effect estimates are negative, consistent 
with the null/negative ATE. TypeA personalized campaigns do not 
outperform TypeBC non-personalized campaigns on daily spend for 
any demographic subgroup in this study population.

However, significant heterogeneity exists in the magnitude of 
underperformance (std_point = 0.162):

**Age group** is the strongest driver of heterogeneity (34% 
feature importance). Age Group 1 shows the most negative CATE 
(-0.42) while Age Group 5-6 show the least negative (-0.02 
to -0.05). Older households are relatively more responsive to 
personalized campaigns.

**Income level** is the second strongest driver (31%). Higher 
income levels show less negative CATE, suggesting wealthier 
households are less disadvantaged by receiving TypeA vs TypeBC.

### Business Recommendation

TypeBC non-personalized campaigns appear more effective at driving 
spend across all segments. If personalized campaigns are to be 
used, they should be targeted at older, higher-income households 
where the performance gap with TypeBC is smallest.

The result may reflect the structural advantage of TypeBC — 
households receiving all available coupons have more opportunities 
to spend than TypeA households receiving 16 targeted coupons, 
regardless of how well those 16 are personalized.

Proceeding to refutation tests to validate robustness of estimates.